# ارزیابی پایپ‌لاین واقعیت افزوده

دو پرسش، که به دو شکل متفاوت شکست می‌خورند:

1. **مقیاس** — فاصله‌ی بین اندازه‌ای که در دیتابیس نوشته شده و اندازه‌ای که در
   خودِ فایل `.glb` پخته شده. این نصفِ خودکارشدنیِ بند ۳-۱ است و باید دقیق
   باشد؛ اندازه حساب است، نه تخمین. نصف دیگر — خطا در برابر متر روی کف واقعی —
   کار کارفرماست (بند ۱۲) و هیچ کدی جایش را نمی‌گیرد. کاری که این نوت‌بوک
   می‌کند این است که آن اندازه‌گیری را **تفسیرپذیر** کند: اگر فایل تا میکرومتر
   درست باشد و گوشی ۴٪ خطا بدهد، خطا در جلسه‌ی AR است نه در پایپ‌لاین.
2. **نرخ موفقیت روی ورودی دیده‌نشده** — همان فرش‌ها، ولی از عکس‌های `cover` و
   `gallery` که پایپ‌لاین برایشان ساخته نشده.

In [ ]:
import sys
from pathlib import Path

# The notebooks live beside the backend, not inside it, so that `app` is
# importable exactly the way the server imports it — same package, same module
# state, no copy.
sys.path.insert(0, str(Path.cwd().parent / "backend"))

In [ ]:
from tempfile import TemporaryDirectory

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "backend" / "scripts"))
from app.eval.ar_assets import Summary, run_pipeline_on
from catalog_profiles import PROFILES

SOURCE = Path.cwd().parent / "data" / "catalog-gen"
SHOTS = ("flat", "cover", "gallery")

In [ ]:
by_shot = {shot: [] for shot in SHOTS}
with TemporaryDirectory() as tmp:
    out = Path(tmp)
    for profile in PROFILES:
        for shot in SHOTS:
            found = sorted(SOURCE.glob(f"{profile.slug}__{shot}.*"))
            if not found:
                continue
            for width_cm, length_cm, *_ in profile.variants:
                by_shot[shot].append(
                    run_pipeline_on(
                        found[0], width_cm=width_cm, length_cm=length_cm, out_dir=out,
                        label=f"{profile.slug}-{width_cm}x{length_cm}", shot=shot,
                    )
                )

pd.DataFrame([Summary(runs).as_row(shot) for shot, runs in by_shot.items()])

## چرا «اطمینان تشخیص» و «اندازه‌ی درست» دو ستون جدا هستند

اولین اجرای این اندازه‌گیری هر دو را در یک عدد جمع کرده بود و نتیجه «۰٪ موفقیت»
شد — درباره‌ی مجموعه‌ای از فایل‌ها که **همه‌شان درست بودند**. اطمینان،
*تشخیص‌دهنده‌ی گوشه* را نمره می‌دهد؛ وقتی پایین باشد، تشخیص به کل کادر برمی‌گردد،
و برای عکس تختی که فرش لبه‌به‌لبه پرش کرده، همان جواب درست است. دو چیز متفاوت
با یک عدد گزارش نمی‌شوند.

In [ ]:
everything = Summary([r for runs in by_shot.values() for r in runs])
scale_errors = [r.scale.worst_error for runs in by_shot.values() for r in runs if r.scale]
print(f"n = {everything.n}")
print(f"بدترین خطای مقیاس: {max(scale_errors):.6%}")
print(f"اندازه‌ی درست: {everything.success_rate:.1%}")
print(f"اطمینان تشخیص گوشه: {everything.automatic_rate:.1%}")

# ارزیابی راهنمای اندازه

کل پاسخِ «چه اندازه فرشی جا می‌شود» زیر یک عدد است: هر پیکسل چند متر ارزش
دارد. برگه‌ی A4 آن عدد را از حدس به اندازه‌گیری تبدیل می‌کند، چون ISO 216
ابعادش را دقیق تعریف کرده.

روی صحنه‌های **مصنوعی** سنجیده می‌شود و این انتخاب دلیل دارد: عکس یک اتاق واقعی
حقیقتِ زمینی ندارد، پس خطای محاسبه‌شده در برابرش خطا در برابر یک تخمین دیگر
است. اینجا صفحه‌ی کف، دوربین و جای برگه ساخته می‌شوند، پس مقیاس درست تا دقت
ماشین معلوم است.

In [ ]:
from app.eval.sizing import Report, default_scenes, run

trials = run(default_scenes())
pd.DataFrame([Report(trials).as_row("همه")])

## آنچه دقت را تعیین می‌کند: اندازه‌ی برگه در کادر

تصحیح گوشه تا حدود نیم پیکسل کار می‌کند و اندازه‌گیری بر طول ضلع برگه تقسیم
می‌شود — پس دقت را بزرگیِ برگه در کادر تعیین می‌کند، و آن تنها چیزی است که
خریدار کنترلش می‌کند: نزدیک‌تر بایستد.

In [ ]:
buckets = [(0, 55), (55, 75), (75, 100), (100, 10_000)]
pd.DataFrame(
    [
        Report([t for t in trials if lo <= t.sheet_px < hi]).as_row(
            f"{lo}–{hi if hi < 10_000 else '∞'} px"
        )
        for lo, hi in buckets
    ]
)

In [ ]:
import matplotlib.pyplot as plt

found = [t for t in trials if t.error is not None]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter([t.sheet_px for t in found], [t.error * 100 for t in found], s=14, alpha=0.6)
ax.axhline(3.0, linestyle="--", linewidth=1)
ax.set_xlabel("sheet's longest side (px)")
ax.set_ylabel("scale error (%)")
ax.set_title("A4 reference: accuracy against how large the sheet is in frame")
plt.tight_layout()